# EMSC WebSocket API

**Endpoint:** `wss://www.seismicportal.eu/standing_order/websocket`

**Purpose:** Real-time earthquake events from the European-Mediterranean Seismological Centre (EMSC).  
The WebSocket pushes JSON messages as soon as a new seismic event is registered.

**Used in:** `services/emsc-listener/emsc_listener.py`

> **Note:** Earthquakes do not occur every second. The timeout per event is set to 120 seconds.

In [4]:
# !pip install websocket-client matplotlib --quiet
import sys
!{sys.executable} -m pip install websocket-client matplotlib --quiet

In [ ]:
import json
import websocket
from datetime import datetime, timezone
import requests
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import urllib.parse
import http.client


EMSC_WS_URL = "wss://www.seismicportal.eu/standing_order/websocket"
USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
WORLD_POP = "https://api.worldpop.org/v1"
OVERPASS_HOST     = "overpass-api.de"
OVERPASS_ENDPOINT = "/api/interpreter"
OPENMETEO_URL = "https://archive-api.open-meteo.com/v1/archive"

MAX_Radius_KM = 100.0

EARTH_CIRCUMFERENCE_KM = 40_075
TIMEOUT_SEC = 240  # seconds to wait for an event to arrive

In [6]:
def _worldpop_params(lat: float, lon: float, radius_km: float) -> str:
    """Build the GeoJSON hexagon query string for the WorldPop Stats API."""
    lat_per_km = 360 / EARTH_CIRCUMFERENCE_KM
    lon_per_km = 360 / (EARTH_CIRCUMFERENCE_KM * math.cos(math.radians(lat)))

    coordinates = [[
        [
            lon + radius_km * math.sin(math.radians(60 * i)) * lon_per_km,
            lat + radius_km * math.cos(math.radians(60 * i)) * lat_per_km,
        ]
        for i in range(6)
    ]]

    geojson = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {},
                "geometry": {"type": "Polygon", "coordinates": coordinates},
            }
        ],
    }
    return f"dataset=wpgppop&year=2020&geojson={json.dumps(geojson)}"


def fetch_worldpop(lat: float, lon: float, radius_km: float) -> dict:
    """WorldPop: population density within radius_km of epicenter (2020)."""
    try:
        resp = requests.get(
            "https://api.worldpop.org/v1/services/stats",
            params=_worldpop_params(lat, lon, radius_km),
            timeout=100,
        )
        if resp.status_code == 200:
            task_id = resp.json().get("taskid", "")
            # print(f"WorldPop task submitted, task ID: {task_id}")
            resp = requests.get(
                f"https://api.worldpop.org/v1/tasks/{task_id}",
                timeout=100,
            )
            return resp.json().get("data", {}).get("total_population", "")
    except Exception as e:
        print(f"WorldPop fetch failed: {e}", flush=True)
        return None

In [7]:
def fetch_usgs_nearby(lat: float, lon: float, radius_km: float):
    """USGS: historical M4.0+ earthquakes within radius_km since 2014."""
    try:
        resp = requests.get(
            USGS_URL,
            params={
                "format":       "geojson",
                "latitude":     lat,
                "longitude":    lon,
                "maxradiuskm":  radius_km,
                "minmagnitude": 4.0,
                "starttime":    "2014-01-01",
                "limit":        1000,
            },
            timeout=15,
        )
        features   = resp.json().get("features", [])

        return [
            {
                "id":    f.get("id"),
                "mag":   f["properties"].get("mag"),
                "time":  f["properties"].get("time"),
                "place": f["properties"].get("place"),
                "coordinates": 
                {
                    "lon": f["geometry"]["coordinates"][0],
                    "lat": f["geometry"]["coordinates"][1],
                    "depth_km": f["geometry"]["coordinates"][2],
                }
            }
            for f in features
        ]

        # magnitudes = [f["properties"]["mag"] for f in features if f["properties"].get("mag")]
        # return {
        #     "count":         len(features),
        #     "max_magnitude": max(magnitudes) if magnitudes else None,
        # }
    except Exception as e:
        print(f"USGS fetch failed: {e}", flush=True)
        return {}

In [22]:
def fetch_overpass(lat: float, lon: float, radius_km: float) -> dict:
    try:
        radius = int(radius_km * 1000)  # convert km to meters
        query = (
            "[out:json][timeout:60];\n"
            "(\n"
            f'  nwr["amenity"="hospital"](around:{radius},{lat},{lon});\n'
            f'  nwr["aeroway"="aerodrome"](around:{radius},{lat},{lon});\n'
            f'  nwr["amenity"="police"](around:{radius},{lat},{lon});\n'
            ");\n"
            "out tags;"
        )
        body = urllib.parse.urlencode({"data": query}).encode("utf-8")
        conn = http.client.HTTPConnection(OVERPASS_HOST, 80, timeout=60)
        conn.request("POST", OVERPASS_ENDPOINT, body=body, headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "User-Agent":   "EarthquakeMonitor/1.0",
        })
        resp = conn.getresponse()
        if resp.status != 200:
            raise RuntimeError(f"Overpass API returned HTTP {resp.status}: {resp.reason}")
        return json.loads(resp.read().decode("utf-8")).get("elements", [])
    except Exception as e:
        print(f"Overpass fetch failed: {e}", flush=True)
        return {}

In [9]:
ws = websocket.create_connection(EMSC_WS_URL, timeout=TIMEOUT_SEC)
print("Connected to EMSC WebSocket. Waiting for event...")

message = ws.recv()
event = json.loads(message)
ws.close()

print(json.dumps(event, indent=2))

Connected to EMSC WebSocket. Waiting for event...
{
  "action": "create",
  "data": {
    "type": "Feature",
    "geometry": {
      "type": "Point",
      "coordinates": [
        94.163,
        25.399,
        -10.0
      ]
    },
    "id": "20260513_0000318",
    "properties": {
      "source_id": "1994483",
      "source_catalog": "EMSC-RTS",
      "lastupdate": "2026-05-13T18:21:35.206025Z",
      "time": "2026-05-13T18:05:10.0Z",
      "flynn_region": "MYANMAR-INDIA BORDER REGION",
      "lat": 25.399,
      "lon": 94.163,
      "depth": 10.0,
      "evtype": "ke",
      "auth": "NDI",
      "mag": 2.8,
      "magtype": "m",
      "unid": "20260513_0000318"
    }
  }
}


In [24]:
properties = event.get("data", {}).get("properties", {})
lat = properties.get("lat", None)
lon = properties.get("lon", None)
print(f"Received event: lat={lat}, lon={lon}")

historical_earthquakes = {}
population = None
infrastructure = {}
if lat is not None and lon is not None:
    # historical_earthquakes = fetch_usgs_nearby(lat=lat, lon=lon, radius_km=MAX_Radius_KM)
    # population = fetch_worldpop(lat=lat, lon=lon, radius_km=MAX_Radius_KM)
    infrastructure = fetch_overpass(lat=37.98, lon=23.73, radius_km=MAX_Radius_KM)

# print(json.dumps(infrastructure, indent=2, ensure_ascii=False))
event["historical_earthquakes"] = [eq["id"] for eq in historical_earthquakes]
event["population"] = population
event["infrastructure"] = [eq["id"] for eq in infrastructure]
print(event["infrastructure"])

Received event: lat=25.399, lon=94.163
[428151392, 458252633, 519860120, 533907296, 613126663, 618030702, 618030790, 747740417, 888106115, 890516519, 996622678, 1008013053, 1042046863, 1078552410, 1247419843, 1308961914, 1333772139, 1333772141, 1416240336, 1434326599, 1444851215, 1507261531, 1519762845, 1613229388, 1753654252, 1759658586, 1792987638, 1842889240, 2102683955, 2467823688, 2479591822, 2502436884, 2574746156, 2575675107, 2620920475, 2675528528, 2677425388, 2721096504, 2724288826, 2834702766, 2838312126, 2892284170, 3033585327, 3107735314, 3256763559, 3283682061, 3283688762, 3694329656, 3750006246, 3762309202, 3817812875, 3978723004, 3980711467, 4038838050, 4081111786, 4221460802, 4237224983, 4464612064, 4524880140, 4641723421, 4691731381, 4799602966, 4808278130, 4933971742, 4969034721, 4990337798, 5061798721, 5063533157, 5094514480, 5144433124, 5199710753, 5329013405, 5352985934, 5374946813, 5531060089, 5591132229, 5593355008, 5685537623, 5865582696, 5924046688, 6196637257,

```json
{
    "version": 0.6, 
    "generator": "Overpass API 0.7.62.11 87bfad18", 
    "osm3s": 
    {
        "timestamp_osm_base": "2026-05-13T17:42:59Z", 
        "copyright": "The data included in this document is from www.openstreetmap.org. The data is made available under ODbL."
    }, 
    "elements": []}
```

# Kafka_Consumers

## MongoDB Writer

In [11]:
def build_upsert(event: dict, unid: str) -> tuple:
    filter_doc = {"_id": unid}
    update_doc = {
        "$setOnInsert": {
            "_id":              unid,
            "received_at":      datetime.now(timezone.utc),
            "worldpop_fetched": False,
            "overpass_fetched": False,
            "usgs_fetched":     False,
            "meteo_fetched":    False,
        }
    }
    return filter_doc, update_doc

flagged_data = build_upsert(event, unid)
update_doc = flagged_data[1]
print(json.dumps(str(update_doc), indent=2))

filter_doc = flagged_data[0]
print(json.dumps(str(filter_doc), indent=2))

NameError: name 'unid' is not defined

```json
{"_id": "20260512_0000205"}
{
	"$setOnInsert": {
		"action": "create", 
		"data": {
			"type": "Feature", 
			"geometry": {
				"type": "Point", 
				"coordinates": [116.89, -9.2, -9.0]
            }, 
			"id": "20260512_0000205", 
			"properties": {
				"source_id": "1993780", 
				"source_catalog": "EMSC-RTS", 
				"lastupdate": "2026-05-12T13: 42: 33.567351Z", 
				"time": "2026-05-12T13: 29: 58.0Z", 
				"flynn_region": "LOMBOK REGION, INDONESIA", 
				"lat": -9.2, 
				"lon": 116.89, 
				"depth": 9.0, 
				"evtype": "ke", 
				"auth": "BMKG", 
				"mag": 2.5, 
				"magtype": "m", 
				"unid": "20260512_0000205"
            }
        }, 
		"_id": "20260512_0000205", 
		"received_at": datetime.datetime(2026,5,12,13,59,25,159272, tzinfo=datetime.timezone.utc), 
		"worldpop_fetched": False, 
		"overpass_fetched": False, 
		"usgs_fetched": False, 
		"meteo_fetched": False
    }
}
```

## S3_Bronze_Writer
Path: s3://eq-monitor/bronze/events/{date_path}/{unid}.json

```json
{
    "action": "create",
    "data": {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [
                -178.6374,
                -32.454,
                -35.0
            ]
        },
        "id": "20260419_0000537",
        "properties": {
            "source_id": "1981286",
            "source_catalog": "EMSC-RTS",
            "lastupdate": "2026-05-12T11:45:51.887638Z",
            "time": "2026-04-19T11:26:07.553Z",
            "flynn_region": "SOUTH OF KERMADEC ISLANDS",
            "lat": -32.454,
            "lon": -178.6374,
            "depth": 35.0,
            "evtype": "ke",
            "auth": "NEIC",
            "mag": 4.4,
            "magtype": "mb",
            "unid": "20260419_0000537"
        }
    }
}
```